In [22]:
from approx.approximate import ApprOXimate
from approx.feature_engineering import MaterialFeatureExtractor
from mendeleev.fetch import fetch_table
import pandas as pd
import numpy as np

# Clean Dataset

some compositions are not compatible with the featurisation method

In [23]:
icsd_df = pd.read_csv("ICSD_CrystStrucData.csv")
icsd_df

,HMS,Chemical,Temperature,Pressure,Crystal_System
0,P 4/m m m,Nd1Ba1Mn1Fe1O5.45,293.0,0.101325,tetragonal
1,P 4/m m m,Nd1Ba1Mn1Fe1O5.17,293.0,0.101325,tetragonal
2,P 4/m m m,Nd1Ba1Mn1Fe1O5.09,293.0,0.101325,tetragonal
3,I 4/m m m,Sr2Mn2.275Cr0.725As2O2,300.0,0.101325,tetragonal
4,P 42 m c,Ca1Mn1Ti1.8V0.2O6,293.0,0.101325,tetragonal
...,...,...,...,...,...
2294,F d -3 Z,Mn21.5Rb49Al92Si100O384,294.0,0.101325,cubic
2295,F 4 3 2,Ca6.3Mn3Ga4.4Al1.3O18,293.0,0.101325,cubic
2296,F -4 3 m,Li1In1Cr3.8Mn0.2O8,293.0,0.101325,cubic
2297,F -4 3 m,Li1In1Cr3.6Mn0.4O8,293.0,0.101325,cubic


In [24]:
approx = ApprOXimate()

for formula in icsd_df['Chemical'].tolist():
    try:
        formula_dict = approx.charge_balance(formula, return_format='string')
    except:
        print(formula)

Ca8Mn10.64Si12.32O56H18
Mn2H46O101Si2W24
La6H113O132Mn2V25
Ce6H109O130Mn2V25
Pr6H113O132Mn2V25
Mn6H88O110W19Zn3
H76Mn5.5O102W18.5Sb2
Na10Mn5H148O156W24
Na12Mn1Nb12O88H100
K2.16Mn16Si26.9O75.8H8
K3.68Mn15.904Si25.472O80.4H23.424
Na12Mn1Nb12O90H104
Mn16Si12As3O57H17
Mn28Cs36Al92Si100O384
Mn28.5Si135Al57O411.2H54.4
Mn21.5Rb49Al92Si100O384


these formulas can't be charge balanced therefore they are removed from the dataset. This is because some of the values for an element is very large

In [25]:
def charge_balance_ok(formula):
    try:
        approx.charge_balance(formula, return_format='string')
        return True
    except Exception:
        return False

mask = icsd_df['Chemical'].apply(charge_balance_ok)
icsd_df = icsd_df[mask].reset_index(drop=True)

icsd_df

,HMS,Chemical,Temperature,Pressure,Crystal_System
0,P 4/m m m,Nd1Ba1Mn1Fe1O5.45,293.0,0.101325,tetragonal
1,P 4/m m m,Nd1Ba1Mn1Fe1O5.17,293.0,0.101325,tetragonal
2,P 4/m m m,Nd1Ba1Mn1Fe1O5.09,293.0,0.101325,tetragonal
3,I 4/m m m,Sr2Mn2.275Cr0.725As2O2,300.0,0.101325,tetragonal
4,P 42 m c,Ca1Mn1Ti1.8V0.2O6,293.0,0.101325,tetragonal
...,...,...,...,...,...
2278,F d -3 m S,Mn0.5Zn0.5Fe1.95Sc0.05O4,293.0,0.101325,cubic
2279,F 4 3 2,Ca6.3Mn3Ga4.4Al1.3O18,293.0,0.101325,cubic
2280,F -4 3 m,Li1In1Cr3.8Mn0.2O8,293.0,0.101325,cubic
2281,F -4 3 m,Li1In1Cr3.6Mn0.4O8,293.0,0.101325,cubic


# Splitting the data

In [26]:
unique_formula = icsd_df['Chemical'].unique()
print(f'{len(unique_formula)} unique formulae:\n{unique_formula}')

# Set a random seed to ensure reproducibility across runs
np.random.seed(seed=42)

# Store a list of all unique formulae
all_formulae = unique_formula.copy()

# Define the proportional size of the dataset split
val_size = 0.20
test_size = 0.10
train_size = 1 - val_size - test_size

# Calculate the number of samples in each dataset split
num_val_samples = int(round(val_size * len(unique_formula)))
num_test_samples = int(round(test_size * len(unique_formula)))
num_train_samples = int(round((1 - val_size - test_size) * len(unique_formula)))

# Randomly choose the formula for the validation dataset, and remove those from the unique list
val_formulae = np.random.choice(all_formulae, size=num_val_samples, replace=False)
all_formulae = [f for f in all_formulae if f not in val_formulae]

# Randomly choose the formula for the test dataset, and remove those from the unique formula list
test_formulae = np.random.choice(all_formulae, size=num_test_samples, replace=False)
all_formulae = [f for f in all_formulae if f not in test_formulae]

# The remaining formula will be used for the training dataset
train_formulae = all_formulae.copy()

print('Number of training formulae:', len(train_formulae))
print('Number of validation formulae:', len(val_formulae))
print('Number of testing formulae:', len(test_formulae))

# Split the original dataset into the train/validation/test datasets using the formulae lists above
icsd_train_split_df = icsd_df[icsd_df['Chemical'].isin(train_formulae)]
icsd_val_split_df = icsd_df[icsd_df['Chemical'].isin(val_formulae)]
icsd_test_split_df = icsd_df[icsd_df['Chemical'].isin(test_formulae)]

print(f'train dataset shape: {icsd_train_split_df.shape}')
print(f'validation dataset shape: {icsd_val_split_df.shape}')
print(f'test dataset shape: {icsd_test_split_df.shape}')

# Check if there are no intersecting formulas in the test training and validation datasets

train_formulae = set(icsd_train_split_df['Chemical'].unique())
val_formulae = set(icsd_val_split_df['Chemical'].unique())
test_formulae = set(icsd_test_split_df['Chemical'].unique())

common_formulae1 = train_formulae.intersection(test_formulae)
common_formulae2 = train_formulae.intersection(val_formulae)
common_formulae3 = test_formulae.intersection(val_formulae)

print(f'# of common formulae in intersection 1: {len(common_formulae1)}; common formulae: {common_formulae1}')
print(f'# of common formulae in intersection 2: {len(common_formulae2)}; common formulae: {common_formulae2}')
print(f'# of common formulae in intersection 3: {len(common_formulae3)}; common formulae: {common_formulae3}')

2128 unique formulae:
['Nd1Ba1Mn1Fe1O5.45' 'Nd1Ba1Mn1Fe1O5.17' 'Nd1Ba1Mn1Fe1O5.09' ...
 'Li1In1Cr3.8Mn0.2O8' 'Li1In1Cr3.6Mn0.4O8' 'Li1In1Cr3.5Mn0.5O8']
Number of training formulae: 1489
Number of validation formulae: 426
Number of testing formulae: 213
train dataset shape: (1617, 5)
validation dataset shape: (438, 5)
test dataset shape: (228, 5)
# of common formulae in intersection 1: 0; common formulae: set()
# of common formulae in intersection 2: 0; common formulae: set()
# of common formulae in intersection 3: 0; common formulae: set()


In [27]:
# save these splits so the dataset is repeatable, other featurisation techniques can be used and validated using the same split dataset

icsd_train_split_df.to_csv('control_dataset_splits/ICSD_train_split.csv', index=False)
icsd_val_split_df.to_csv('control_dataset_splits/ICSD_val_split.csv', index=False)
icsd_test_split_df.to_csv('control_dataset_splits/ICSD_test_split.csv', index=False)

# Featurise the dataset using approximate

In [29]:
def featurize_df(df, extractor, formula_col="Compound"):
    formulas = df[formula_col].tolist()
    feature_rows = extractor.featurize_many(formulas)
    feat_df = pd.DataFrame(feature_rows, index=df.index)

    # Keep only rows that featurized successfully
    feature_cols = feat_df.drop(columns=["formula"], errors="ignore")
    if feature_cols.shape[1] == 0:
        return df.iloc[0:0].copy()

    valid_rows = feature_cols.notna().any(axis=1)
    df_valid = df.loc[valid_rows]
    feat_valid = feat_df.loc[valid_rows]

    failed_formulas = df.loc[~valid_rows, formula_col]
    for formula in failed_formulas:
        print("Formula Fail:", formula)

    return df_valid.join(feat_valid)

# Initialize modules
approx = ApprOXimate()
ptable = fetch_table("elements")
extractor = MaterialFeatureExtractor(approx, ptable, mode="all")

# Run featurization
df_feat = featurize_df(icsd_df, extractor, formula_col="Chemical")
df_feat.head()

,HMS,Chemical,Temperature,Pressure,Crystal_System,formula,all_valence_s_sum,all_valence_s_avg,all_valence_s_dev,all_valence_s_min,...,all_gordy_en_max,all_gordy_en_range,all_gordy_en_mode,all_mb_en_sum,all_mb_en_avg,all_mb_en_dev,all_mb_en_min,all_mb_en_max,all_mb_en_range,all_mb_en_mode
0,P 4/m m m,Nd1Ba1Mn1Fe1O5.45,293.0,0.101325,tetragonal,Nd1Ba1Mn1Fe1O5.45,10.90,1.153439,0.988158,0.0,...,0.465308,0.424263,0.041044,0.460741,0.048756,0.025544,0.0,0.063694,0.063694,0.063492
1,P 4/m m m,Nd1Ba1Mn1Fe1O5.17,293.0,0.101325,tetragonal,Nd1Ba1Mn1Fe1O5.17,10.34,1.127590,0.991827,0.0,...,0.465308,0.424263,0.041044,0.443294,0.048342,0.025803,0.0,0.063694,0.063694,0.063492
2,P 4/m m m,Nd1Ba1Mn1Fe1O5.09,293.0,0.101325,tetragonal,Nd1Ba1Mn1Fe1O5.09,10.18,1.119912,0.992785,0.0,...,0.465308,0.424263,0.041044,0.438309,0.048219,0.025878,0.0,0.063694,0.063694,0.063492
3,I 4/m m m,Sr2Mn2.275Cr0.725As2O2,300.0,0.101325,tetragonal,Sr2Mn2.275Cr0.725As2O2,8.00,0.380952,0.785353,0.0,...,0.378548,0.337504,0.203202,1.119075,0.053289,0.039683,0.0,0.166667,0.166667,0.042553
4,P 42 m c,Ca1Mn1Ti1.8V0.2O6,293.0,0.101325,tetragonal,Ca1Mn1Ti1.8V0.2O6,12.00,1.200000,0.979796,0.0,...,0.252836,0.211791,0.041044,0.435277,0.043528,0.028172,0.0,0.063492,0.063492,0.063492


In [30]:
import pandas as pd

def clean_feature_table(
    df,
    keep_cols=None,
    drop_formula_col=False,
    drop_duplicate_columns=True,
    verbose=True,
):
    """
    Initial safe cleanup for a featurized dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Full dataframe including metadata + features.
    keep_cols : list[str] | None
        Columns to preserve even if they are constant.
        Example: ["StructuredFormula", "split", "crystal_system"]
    drop_formula_col : bool
        If True, drop the generated 'formula' column from the feature table.
    drop_duplicate_columns : bool
        If True, remove duplicate feature columns with identical values.
    verbose : bool
        Print summary of dropped columns.

    Returns
    -------
    df_clean : pd.DataFrame
        Cleaned dataframe.
    summary : dict
        Summary of what was removed.
    """
    df_clean = df.copy()
    keep_cols = keep_cols or []

    removed = {
        "all_null": [],
        "constant": [],
        "duplicate": [],
        "manual": [],
    }

    if drop_formula_col and "formula" in df_clean.columns:
        df_clean = df_clean.drop(columns=["formula"])
        removed["manual"].append("formula")

    # Drop all-null columns
    all_null_cols = [
        col for col in df_clean.columns
        if col not in keep_cols and df_clean[col].isna().all()
    ]
    if all_null_cols:
        df_clean = df_clean.drop(columns=all_null_cols)
        removed["all_null"] = all_null_cols

    # Drop constant columns
    constant_cols = [
        col for col in df_clean.columns
        if col not in keep_cols and df_clean[col].nunique(dropna=False) <= 1
    ]
    if constant_cols:
        df_clean = df_clean.drop(columns=constant_cols)
        removed["constant"] = constant_cols

    # Drop duplicate columns
    if drop_duplicate_columns:
        feature_cols = [col for col in df_clean.columns if col not in keep_cols]
        duplicate_cols = []
        seen = {}

        for col in feature_cols:
            signature = tuple(df_clean[col].fillna("__NA__").tolist())
            if signature in seen:
                duplicate_cols.append(col)
            else:
                seen[signature] = col

        if duplicate_cols:
            df_clean = df_clean.drop(columns=duplicate_cols)
            removed["duplicate"] = duplicate_cols

    summary = {
        "n_start_cols": df.shape[1],
        "n_end_cols": df_clean.shape[1],
        "n_removed_total": df.shape[1] - df_clean.shape[1],
        "removed": removed,
    }

    if verbose:
        print(f"Started with {summary['n_start_cols']} columns")
        print(f"Ended with {summary['n_end_cols']} columns")
        print(f"Removed {summary['n_removed_total']} columns")
        print(f"  all-null: {len(removed['all_null'])}")
        print(f"  constant: {len(removed['constant'])}")
        print(f"  duplicate: {len(removed['duplicate'])}")
        print(f"  manual: {len(removed['manual'])}")

    return df_clean, summary

keep_cols = [
    "Chemical",
    "Crystal_System",
    "split"
]

df_feat_clean, cleanup_summary = clean_feature_table(
    df_feat,
    keep_cols=keep_cols,
    drop_formula_col=False,
    drop_duplicate_columns=True,
    verbose=True,
)

Started with 237 columns
Ended with 191 columns
Removed 46 columns
  all-null: 0
  constant: 37
  duplicate: 9
  manual: 0


In [31]:
# Before cleaning
print("NaNs before:", df_feat.isna().sum().sum())

# After cleaning
print("NaNs after:", df_feat_clean.isna().sum().sum())

# Which columns still have NaNs
nan_summary = df_feat_clean.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)
print(nan_summary)

NaNs before: 318
NaNs after: 318
all_pauling_en_sum      53
all_pauling_en_avg      53
all_pauling_en_dev      53
all_pauling_en_min      53
all_pauling_en_max      53
all_pauling_en_range    53
dtype: int64


# Final Save

### Approximate is split now the same way the above splits are like

In [32]:
df_feat_train_clean = df_feat_clean[df_feat_clean['Chemical'].isin(train_formulae)]
df_feat_val_clean = df_feat_clean[df_feat_clean['Chemical'].isin(val_formulae)]
df_feat_test_clean = df_feat_clean[df_feat_clean['Chemical'].isin(test_formulae)]

print(f'train dataset shape: {df_feat_train_clean.shape}')
print(f'validation dataset shape: {df_feat_val_clean.shape}')
print(f'test dataset shape: {df_feat_test_clean.shape}')

train dataset shape: (1617, 191)
validation dataset shape: (438, 191)
test dataset shape: (228, 191)


In [33]:
feat_formulae = set(df_feat['Chemical'].unique())
icsd_formulae = train_formulae | val_formulae | test_formulae

only_in_feat = feat_formulae - icsd_formulae
only_in_icsd = icsd_formulae - feat_formulae

print(f'Formulae only in df_feat (will be dropped): {len(only_in_feat)}')
print(f'Formulae only in icsd_df (missing from df_feat): {len(only_in_icsd)}')

Formulae only in df_feat (will be dropped): 0
Formulae only in icsd_df (missing from df_feat): 0


In [34]:
df_feat_train_clean.to_csv('approx_dataset_splits/app_train_split.csv', index=False)
df_feat_val_clean.to_csv('approx_dataset_splits/app_val_split.csv', index=False)
df_feat_test_clean.to_csv('approx_dataset_splits/app_test_split.csv', index=False)